# Day 63：毕业设计测试、Ragas 评估报告与性能/成本压测优化

> **项目**：星辰数科 · 智枢企业级智能体平台 (`v1.0.0-day63`)
> **目标**：通过离线 Ragas 四维指标验证质量门禁，并对比 `Prompt Caching + Rerank Top-K + Model Routing` 三重优化策略的 P95 延迟与千次请求成本。

## 1. 环境初始化与加载 `Day 63` 评估模块

In [1]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "project_iterations" / "day_63").exists():
    REPO_ROOT = REPO_ROOT.parents[2]

DAY63_DIR = REPO_ROOT / "project_iterations" / "day_63"
if str(DAY63_DIR) not in sys.path:
    sys.path.insert(0, str(DAY63_DIR))

from zhishu_agent.capstone_eval import (
    CapstoneEvaluator,
    OptimizationAdvisor,
    build_sample_capstone_cases,
)

cases = build_sample_capstone_cases()
print(f"成功加载 {len(cases)} 条智枢平台 Capstone 评估基准用例。")

成功加载 6 条智枢平台 Capstone 评估基准用例。


## 2. 运行 Baseline 离线 Ragas 评估与性能基准测试

In [2]:
evaluator = CapstoneEvaluator()
baseline_report = evaluator.evaluate_suite(cases, profile_name="Baseline")
print(baseline_report.to_markdown())

# 智枢智能体平台评估报告（Baseline · v1.0.0-day63）

- **样本总数**: `6`
- **质量门禁状态**: **PASS ✅** (综合 Ragas 分数 `0.7640`)

## 1. Ragas 质量指标

| 指标 | 得分 | 说明 |
| :--- | :---: | :--- |
| Faithfulness (忠实度) | `0.5919` | 生成内容被检索上下文支撑的比例 |
| Answer Relevancy (相关性) | `0.6445` | 回答与问题及标准答案的对齐度 |
| Context Precision (上下文精确率) | `0.9722` | 排序靠前检索片段的证据纯度 |
| Citation Accuracy (引用准确率) | `1.0000` | 溯源引用 ID 的 F1 准确率 |
| **Overall Ragas Score** | **`0.7640`** | 四维核心指标综合均值 |

## 2. 性能与 Token 成本基准

| 基准项 | 数值 |
| :--- | :---: |
| P50 延迟 | `1700.00 ms` |
| P95 延迟 | `2300.00 ms` |
| 平均每次请求 Token | `2242.5` |
| Prompt Cache 命中率 | `0.00%` |
| 千次请求成本 (CNY) | `¥53.48` |


## 3. 运行 `OptimizationAdvisor` 对比三重优化策略与月度 ROI

In [3]:
advisor = OptimizationAdvisor(evaluator)
comparison = advisor.compare(
    cases,
    enable_prompt_caching=True,
    rerank_top_k=2,
    enable_model_routing=True,
)

assert comparison.optimized.passed_quality_gate is True
assert comparison.optimized.overall_ragas_score >= comparison.baseline.overall_ragas_score
assert comparison.cost_reduction_pct >= 40.0

print(comparison.to_markdown())
print("\n[核心对比 JSON]")
print(json.dumps({
    "baseline_ragas": comparison.baseline.overall_ragas_score,
    "optimized_ragas": comparison.optimized.overall_ragas_score,
    "latency_p95_reduction_pct": comparison.latency_p95_reduction_pct,
    "cost_reduction_pct": comparison.cost_reduction_pct,
    "monthly_savings_cny_at_100k_req": comparison.monthly_savings_cny_at_100k_req,
}, ensure_ascii=False, indent=2))

# 智枢平台性能与成本优化对比报告（v1.0.0-day63）

## 1. 核心指标对比（Baseline vs Optimized）

| 维度 | Baseline | Optimized | 变化幅度 |
| :--- | :---: | :---: | :---: |
| 综合 Ragas 评分 | `0.7640` | `0.7697` | `+0.0057` |
| P95 延迟 | `2300.00 ms` | `1661.52 ms` | `-27.76%` |
| 单请求平均 Token | `2242.5` | `1962.7` | `-12.48%` |
| Cache 命中率 | `0.00%` | `55.00%` | `+55.00%` |
| 千次请求成本 | `¥53.48` | `¥21.38` | `-60.02%` |

> **ROI 预估**：按月均 100,000 次企业调用计算，优化后预计每月节省模型成本 **`¥3209.86`**。

## 2. 落地优化建议清单

- **[P0] Prompt Caching（系统提示词与制度前缀缓存）**：对智枢平台固定 System Prompt、合规红线清单与高频财报摘要前缀启用 KV Cache 复用。 （预计降本 `38.0%`，降延迟 `16.0%`，质量影响：无损（严格等价推理））
- **[P0] Model Routing（简单/复杂意图分级路由）**：通过轻量意图分类器将常规制度问答与单指标查询路由至 Lite 模型，复杂多跳财报分析保留 Flagship 模型。 （预计降本 `32.0%`，降延迟 `35.0%`，质量影响：可控（简单任务准确率持平））
- **[P1] Rerank Top-2 精排裁剪**：混合检索后经 Cross-Encoder 精排仅保留 Top-2 高相关片段，剔除尾部噪声上下文。 （预计降本 `20.0%`，降延迟 `14.0%`，质量影响：正向提升（Context Precision 显著提高））

[核心对比 JSON]
{
  "baseline_ragas": 0.764,
  "optimized_ragas": 0.7697,
  "latency_p95_reduction_pct": 27.76,
  "cost